In [ ]:
###--- Load libraries and set the location path for analysis data ---###

In [ ]:
# Environment setup
import numpy as np
import scanpy as sc
import pandas as pd
import scipy.io
import matplotlib as mpl
import batchglm.api as glm
import diffxpy.api as de
import decoupler as dc

from matplotlib import rcParams
import bbknn
import os
import sys
import scipy
import seaborn as sns
import scipy.io as sio
import scanpy.external as sce
import matplotlib.pyplot as plt
import scipy.sparse as sp

In [ ]:
sc.settings.verbosity = 2  # show logging output
sc.settings.dir = "scRNA_Glyco/sc_lymp"
sc.settings.autosave = True  # save figures, do not show them
sc.settings.figdir = "scRNA_Glyco/sc_lymfo/figure"
sc.settings.set_figure_params(dpi=200, format="pdf", dpi_save=800) # set sufficiently high resolution for saving 400dpi

In [ ]:
###--- Load pre-filtering data ---###

In [ ]:
# Data Loading
adata_celltypist = sc.read("sc_celltypist_lymphoid.h5ad")
# Start with Raw data
adata_celltypist.X = adata_celltypist.layers["counts"].copy()

In [ ]:
###--- Normalization, Highly Variable Gene, scale ---###

In [ ]:
# Normalization
sc.pp.normalize_total(adata_celltypist, target_sum=1e4)
adata_celltypist.layers["norm10k"] = adata_celltypist.X

In [ ]:
# Logarithmize the data:
sc.pp.log1p(adata_celltypist)
adata_celltypist.layers["log1p"] = adata_celltypist.X

In [ ]:
# Highly Variable Gene Selection
sc.pp.highly_variable_genes(adata_celltypist, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pl.highly_variable_genes(adata_celltypist)

In [ ]:
#Freeze the state of the AnnData object
adata_celltypist.raw = adata_celltypist
adata_celltypist.raw.var.index.is_unique

In [ ]:
adata_celltypist

In [ ]:
# Scale each gene to unit variance. 
sc.pp.scale(adata_celltypist, max_value=10)

In [ ]:
###--- Integration with Harmony ---###

In [ ]:
#Data visualization: Clustering

In [ ]:
# Dimensionality Reduction: Reduce the dimensionality of the data by running principal component analysis (PCA), which reveals the main axes of variation and denoises the data.
sc.pp.pca(adata_celltypist, svd_solver="arpack", use_highly_variable=True,  n_comps=60)
sc.pl.pca_variance_ratio(adata_celltypist, log=True, n_pcs=60)

In [ ]:
#Computing the neighborhood graph
sc.pp.neighbors(adata_celltypist, n_neighbors=20, n_pcs=60)
sc.tl.umap(adata_celltypist)

In [ ]:
# Clustering the neighborhood graph
sc.tl.leiden(adata_celltypist, key_added="leiden", resolution=1.1)

In [ ]:
sc.pl.umap(adata_celltypist, color=['leiden','donor', 'label', 'sample'], wspace=0.5, save='_batch_before_harmony_lym')

In [ ]:
#Running Harmony
sce.pp.harmony_integrate(adata_celltypist, 'donor')

In [ ]:
adata_celltypist.obsm['X_pca'] = adata_celltypist.obsm['X_pca_harmony']

In [ ]:
sc.pp.neighbors(adata_celltypist, n_neighbors=20, n_pcs = 60)
sc.tl.umap(adata_celltypist)

In [ ]:
sc.tl.leiden(adata_celltypist, resolution=1)

In [ ]:
sc.pl.umap(adata_celltypist, color=['leiden','donor', 'label', 'sample'],   wspace=0.5, save='_batch_after_harmony_lym')

In [ ]:
#Data visualization: Clustering

In [ ]:
umap_cluster_col = ['#E27877', '#BE4B3F', '#863221', '#DE4927', '#C17E5D', '#895632', '#EB8D36', '#FACE5B', '#C2A736', '#8C8036', '#F9EE85', '#BFC64C', '#808D3D', '#C1D342', '#A9C662', '#6E8E3F', '#A4C745', '#6BB24A', '#658C4B', '#99C249', '#63AD4B', '#388A41', '#99C36E', '#63AD4F', '#3A8C45', '#70B54B', '#72B77D', '#418C54', '#79B771', '#56AE68', '#528C72', '#338C7C', '#56B498', '#83C4AE', '#518C73', '#78B770']
leiden_colors = {str(i): umap_cluster_col[i] for i in range(36)}

In [ ]:
sc.pl.umap(adata_celltypist, color=['leiden'], save='_cluster_lym_on', legend_loc='on data',  palette=leiden_colors)

In [ ]:
###--- IDENTIFYING CELLULAR STRUCTURE ---###

In [ ]:
#- Differentially expressed genes: TOP MARKER -#
sc.tl.rank_genes_groups(adata_celltypist, groupby='leiden', reference='rest', method='wilcoxon' , key_added="dea_leiden", pts=True)

top_markers = pd.DataFrame(adata_celltypist.uns['dea_leiden']['names']).head(100)
print(top_markers)

In [ ]:
# save results
result = adata_celltypist.uns['dea_leiden']
groups = result['names'].dtype.names
result_df = pd.DataFrame(
    {group + '_' + key[:10]: result[key][group]
    for group in groups for key in ['names', 'scores', 'logfoldchanges', 'pvals','pvals_adj','pts','pts_rest']}).head(100)
result_df
result_df.to_csv("differential_expression_results_All_lymphoid_clusters.csv", index=False)

In [ ]:
#LYMPHOID UMAP MARKER GENES: 
marker_genes_dict_Lyn = {
'Proliferative' : [ 'MKI67' , 'TOP2A','STMN1'],    
'Lineage B' : ['CD19','MS4A1'],
'Pro Pre B' : ['CD34','TCF4'],
'Immature B' : ['CD79A' , 'IGHM'],
'Mature B' : ['IGHD'],
'Plasma cells' : ['JCHAIN' , 'IGHA1' , 'IGKC' , 'TXNDC5' , 'XBP1' , 'HSP90B1'],
'CD8+ T cells' : ['CD3E' ,  'CD8A', 'IFNG','GZMA'],
'NK1' : ['NCAM1' , 'IL18R1'],
'NK2' : ['PRF1','GZMB']
}

In [ ]:
sc.pl.dotplot(
    adata_celltypist,
    var_names=marker_genes_dict_Lyn,
    groupby='leiden',
    dendrogram=True,
    color_map='viridis',
    vmin=0,
    vmax=1,
    standard_scale='var',
    save='_marker_specificgenes_family_cluster_Lyn'
)

In [ ]:
# From markers to cluster cell_type annotation refine
cell_name = {'0': 'Immature B',
'1': 'Pre-B',
'2': 'Mature B',            
'3': 'Immature B',
'4': 'Immature B',
'5': 'Proliferative',
'6': 'Mature B',
'7': 'Immature B',
'8': 'NK2 cells',                
'9': 'Mature B',
'10': 'Pro-B',
'11': 'CD8+ T cells',             
'12': 'Mature B',         
'13': 'NK1 cells',             
'14': 'Proliferative',              
'15': 'Plasma cells'}

adata_celltypist.obs["subcell_type"] = adata_celltypist.obs.leiden.map(cell_name)

In [ ]:
adata_celltypist.obs['subcell_type'] = pd.Categorical(adata_celltypist.obs['subcell_type'])
print(adata_celltypist.obs['subcell_type'].cat.categories)

In [ ]:
# Save the state of the AnnData object Counts.
adata_pp = adata_celltypist.copy()
adata_celltypist.write("sc_lymphoid_clustering.h5ad")

In [ ]:
###--- Evaluation of cell typy distribution ---###

In [ ]:
adata_celltypist.obs['sample']

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# List of annotation categories
comparison_id= ['CAR1','CAR2','CAR3','Tr2DG1','Tr2DG2','Tr2DG3','TrTUN1','TrTUN2','TrTUN3']

# Create a boolean series indicating whether each annotation is in the list of desired categories
boolean_mask = adata_celltypist.obs['sample'].isin(comparison_id)
subset_adata_ly_cmp = adata_celltypist[boolean_mask, :]

# Create the cross-tabulated data
tmp = pd.crosstab( subset_adata_ly_cmp.obs['label'], subset_adata_ly_cmp.obs['leiden'], normalize='index')

# Create the horizontal stacked bar plot
ax = tmp.plot(kind='bar',stacked=True, edgecolor='none',  color=leiden_colors)

# Define labels and legend
horiz_offset = 1.03
vert_offset = 1.
ax.legend(bbox_to_anchor=(horiz_offset, vert_offset))
ax.set_ylabel("Normalized Counts")
ax.set_xlabel("Sample")
ax.set_title("Normalized Stacked Bar Plot of Annotation Categories")

# Save the plot
plt.savefig('scRNA_Glyco/figure/lynfo_categories_barplot_all_clusters.pdf', bbox_inches='tight')

# Show the plot
plt.show()